# Halfedge mesh geometry
Halfedge-based geometry implementation. Optional visualisation and external mesh inputs are described in README.md.


In [ ]:
# =============================================================================
# SETUP: Import from the provided library and basic setup
# =============================================================================
import sys
import math
import numpy as np
from sklearn.neighbors import KDTree
import os
import copy

# Ensure the library is available. Assuming it's saved as 'halfedge_mesh.py' in the same directory.
try:
    from halfedge_mesh import HalfedgeMesh, Vertex, Facet, Halfedge
except ImportError:
    print("Error: Could not import 'halfedge_mesh.py'. Please ensure it exists.")
    # For the notebook context, we might not exit, but subsequent cells will fail.

# --- Monkey Patching Facet.get_center as requested for Task 1.2 ---
# The provided library might not have get_center implemented for Facet.
def facet_get_center(self):
    verts = self.get_vertices()
    if not verts: return [0,0,0]
    coords = np.array([v.get_vertex() for v in verts])
    return np.mean(coords, axis=0).tolist()

Facet.get_center = facet_get_center

# --- Output directory ---
OUTPUT_DIR = "output_models"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- Visualization check ---
try:
    import open3d as o3d
    HAS_O3D = True
except ImportError:
    HAS_O3D = False
    print("Warning: Open3D is not installed.")

# --- Helper function for saving OBJ ---
def save_obj_from_mesh(mesh, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, 'w') as f:
        # Reset indices just in case
        for i, v in enumerate(mesh.vertices):
            v.index = i
            xyz = v.get_vertex()
            f.write(f"v {xyz[0]} {xyz[1]} {xyz[2]}\n")
        for f_obj in mesh.facets:
            verts = f_obj.get_vertices()
            if not verts: continue
            indices_str = " ".join([str(v.index + 1) for v in verts])
            f.write(f"f {indices_str}\n")
    print(f"Saved: {filepath}")

# --- Helper for visualization ---
def visualize_superposition(source_pts, target_pts, title="Overlay"):
    if not HAS_O3D: return
    pcd_src = o3d.geometry.PointCloud()
    pcd_src.points = o3d.utility.Vector3dVector(source_pts)
    pcd_tgt = o3d.geometry.PointCloud()
    pcd_tgt.points = o3d.utility.Vector3dVector(target_pts)
    
    pcd_tgt.paint_uniform_color([1, 0, 0]) # Target = Red
    if "Start" in title:
        pcd_src.paint_uniform_color([1, 0.6, 0]) # Start = Orange
    else:
        pcd_src.paint_uniform_color([0, 1, 0])   # End = Green

    pcd_tgt.estimate_normals()
    pcd_src.estimate_normals()
    
    print(f" >> [Visualizing] {title}")
    o3d.visualization.draw_geometries([pcd_src, pcd_tgt], window_name=title, width=800, height=600, point_show_normal=False)

In [ ]:
# =============================================================================
# =============================================================================

def generate_cube():
    """Generates a simple cube with quad faces using the halfedge structure."""
    # Define 8 vertices for a cube centered at origin with side length 2
    coords = [[-1,-1,1],[1,-1,1],[1,1,1],[-1,1,1],[-1,-1,-1],[1,-1,-1],[1,1,-1],[-1,1,-1]]
    # Define 6 quad faces (CCW winding)
    faces_indices = [[0,1,2,3],[1,5,6,2],[5,4,7,6],[4,0,3,7],[3,2,6,7],[4,5,1,0]]
    
    # Initialize vertices using the library's Vertex class
    vertices = [Vertex(c[0],c[1],c[2],i) for i,c in enumerate(coords)]
    facets = []
    edges = {} 
    halfedges_list = []

    for f_idx, indices in enumerate(faces_indices):
        facet = Facet(-1, -1, -1, index=f_idx)
        facets.append(facet)
        n = len(indices)
        face_edges = []
        
        # Create halfedges for the face
        for i in range(n):
            u, v = indices[i], indices[(i+1)%n]
            if (u,v) not in edges: edges[(u,v)] = Halfedge()
            he = edges[(u,v)]
            he.vertex = vertices[u] 
            he.facet = facet
            vertices[u].halfedge = he 
            face_edges.append(he)
            halfedges_list.append(he)

        # Link halfedges within the face (next/prev)
        facet.halfedge = face_edges[0]
        for i in range(n):
            curr = face_edges[i]
            curr.next = face_edges[(i+1)%n]
            curr.prev = face_edges[(i-1)%n]
            
            # Link opposites if the neighboring edge already exists
            u_curr, v_curr = indices[i], indices[(i+1)%n]
            if (v_curr, u_curr) in edges:
                curr.opposite = edges[(v_curr, u_curr)]
                edges[(v_curr, u_curr)].opposite = curr

    # Explicitly pass empty lists to avoid mutable default argument issues
    mesh = HalfedgeMesh(vertices=[], halfedges=[], facets=[])
    mesh.vertices = vertices
    mesh.facets = facets
    mesh.halfedges = halfedges_list
    mesh.edges = edges 
    return mesh

# --- Execution for Task 1.1 ---
cube = generate_cube()
save_obj_from_mesh(cube, "cube.obj")

# Output the counts as requested
print(f"Task 1.1 Completed: Cube generated successfully.")
print(f"Number of Vertices: {len(cube.vertices)}")
print(f"Number of Faces:    {len(cube.facets)}")
print(f"Number of Halfedges:{len(cube.halfedges)}")

In [ ]:
# =============================================================================
# TASK 1.2: Calculate facet center - VISUALIZATION FIXED (Direct conversion)
# =============================================================================

print("--- Task 1.2: Facet Center Calculation ---")

center_points = []
for f in cube.facets:
    center = f.get_center()
    center_points.append(center)
    print(f"Facet {f.index:02d} center: {center}")

# --- Visualization Logic ---
if HAS_O3D:
    print("\n[Visualizing] Cube (Wireframe) with Facet Centers...")
    print("[Action]: Please Screenshot, then CLOSE the window to continue.")
    
    visual_elements = []

    # 1. Manually build a LineSet from the HalfedgeMesh (Handles Quads correctly)
    points = [v.get_vertex() for v in cube.vertices]
    lines = []
    
    for f in cube.facets:
        # Get vertex indices for this face
        verts = f.get_vertices()
        indices = [v.index for v in verts]
        n = len(indices)
        # Add edges for the face loop (0-1, 1-2, 2-3, 3-0)
        for i in range(n):
            lines.append([indices[i], indices[(i+1)%n]])
            
    # Create Open3D LineSet
    lineset = o3d.geometry.LineSet()
    lineset.points = o3d.utility.Vector3dVector(points)
    lineset.lines = o3d.utility.Vector2iVector(lines)
    lineset.paint_uniform_color([0, 0, 0]) # White lines (High contrast)
    visual_elements.append(lineset)

    # 2. Add Red Spheres for centers
    for c in center_points:
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.1)
        sphere.translate(c)
        sphere.paint_uniform_color([1, 0, 0]) # Red
        sphere.compute_vertex_normals()
        visual_elements.append(sphere)
    
    # Draw
    o3d.visualization.draw_geometries(visual_elements, 
                                      window_name="Task 1.2 Facet Centers",
                                      width=800, height=600)
else:
    print("Open3D not installed, skipping visualization.")

In [ ]:
# =============================================================================
# =============================================================================

def compute_dual(mesh):
    """Computes the dual of a halfedge mesh."""
    # Initialize new mesh with empty lists
    dual = HalfedgeMesh(vertices=[], halfedges=[], facets=[])
    dual.edges = {} 
    pf_to_dv = {}
    
    # 1. Dual Vertices = Primal Facet Centers
    for pf in mesh.facets:
        c = pf.get_center()
        dv = Vertex(c[0], c[1], c[2], len(dual.vertices))
        dual.vertices.append(dv)
        pf_to_dv[pf.index] = dv
        
    edges = {}
    dual_halfedges = []

    # 2. Dual Facets = Primal Vertices
    for pv in mesh.vertices:
        start_he = pv.halfedge
        if start_he is None: continue 
        
        curr = start_he
        surrounding_face_indices = []
        count = 0 
        
        # Traverse around the vertex to find all incident faces
        # Logic: h = h.prev.opposite (assuming he.vertex is origin)
        while True:
            if curr.facet is not None:
                surrounding_face_indices.append(curr.facet.index)
            
            if curr.prev and curr.prev.opposite:
                curr = curr.prev.opposite
            else:
                break 
            
            if curr == start_he or count > 100: break
            count += 1
            
        if len(surrounding_face_indices) < 3: continue
        
        df = Facet(-1, -1, -1, index=len(dual.facets))
        dual.facets.append(df)
        
        n = len(surrounding_face_indices)
        face_hes = []
        dv_indices = [pf_to_dv[f_idx].index for f_idx in surrounding_face_indices]
        
        for i in range(n):
            u, v = dv_indices[i], dv_indices[(i+1)%n]
            if (u,v) not in edges: edges[(u,v)] = Halfedge()
            he = edges[(u,v)]
            he.vertex = dual.vertices[u]
            he.facet = df
            dual.vertices[u].halfedge = he
            face_hes.append(he)
            dual_halfedges.append(he)
            
        df.halfedge = face_hes[0]
        
        for i in range(n):
            face_hes[i].next = face_hes[(i+1)%n]
            face_hes[i].prev = face_hes[(i-1)%n]
            
            curr_he = face_hes[i]
            u_idx = curr_he.vertex.index
            v_idx = curr_he.next.vertex.index
            if (v_idx, u_idx) in edges:
                curr_he.opposite = edges[(v_idx, u_idx)]
                edges[(v_idx, u_idx)].opposite = curr_he
                
    dual.halfedges = dual_halfedges
    dual.edges = edges
    return dual

# --- Execution for Task 1.3 ---
print("Computing Duals...")
dual_cube = compute_dual(cube)
save_obj_from_mesh(dual_cube, "cube_dual.obj")

dd_cube = compute_dual(dual_cube)
save_obj_from_mesh(dd_cube, "cube_double_dual.obj")

# Processing other files if available
test_files = ["halfedge-mesh-test-meshes/sphere1.off", 
              "halfedge-mesh-test-meshes/igea11706.off", 
              "halfedge-mesh-test-meshes/flower.off"]

for fpath in test_files:
    if os.path.exists(fpath):
        print(f"Processing Dual for: {fpath}")
        try:
            mesh = HalfedgeMesh(fpath, vertices=[], halfedges=[], facets=[])
            if mesh.edges is None: mesh.edges = {} # Initialize if None
            
            dual = compute_dual(mesh)
            name = os.path.basename(fpath).split('.')[0]
            save_obj_from_mesh(dual, f"{name}_dual.obj")
            
            dd = compute_dual(dual)
            save_obj_from_mesh(dd, f"{name}_double_dual.obj")
        except Exception as e:
            print(f"Error processing {fpath}: {e}")

In [ ]:
# =============================================================================
# =============================================================================

def compute_volume(mesh):
    """Computes signed volume of the mesh using the first vertex as reference."""
    if not mesh.vertices: return 0.0
    # Use the library's get_vertex() which returns a list [x,y,z]
    p_ref = np.array(mesh.vertices[0].get_vertex())
    vol = 0.0
    for f in mesh.facets:
        verts = f.get_vertices()
        if len(verts) < 3: continue
        v0 = np.array(verts[0].get_vertex())
        for i in range(1, len(verts)-1):
            v1 = np.array(verts[i].get_vertex())
            v2 = np.array(verts[i+1].get_vertex())
            # Tetrahedral volume contribution
            vol += np.dot(np.cross(v0-p_ref, v1-p_ref), v2-p_ref)
    return abs(vol) / 6.0

# --- Execution for Task 1.4 ---
try:
    fpath = "halfedge-mesh-test-meshes/flower.off"
    if os.path.exists(fpath):
        print("\n--- Task 1.4: Flower Double Dual Iterations ---")
        # Explicitly init with empty lists to avoid shared state issues
        flower = HalfedgeMesh(fpath, vertices=[], halfedges=[], facets=[])
        if flower.edges is None: flower.edges = {}
        
        curr = flower
        vol = compute_volume(curr)
        
        # 1. Update Header to include Num Vertices and Num Faces
        print(f"{'Iter':<5} | {'Num V':<8} | {'Num F':<8} | {'Volume':<15} | {'Ratio'}")
        print("-" * 65)
        
        # 2. Update Initial State Print
        print(f"{0:<5} | {len(curr.vertices):<8} | {len(curr.facets):<8} | {vol:<15.6f} | -")
        
        prev_vol = vol
        for i in range(1, 11):
            # Compute Double Dual
            curr = compute_dual(compute_dual(curr))
            
            # Compute Volume
            vol = compute_volume(curr)
            ratio = vol / prev_vol if prev_vol else 0
            
            # 3. Update Loop Print
            print(f"{i:<5} | {len(curr.vertices):<8} | {len(curr.facets):<8} | {vol:<15.6f} | {ratio:.4f}")
            
            prev_vol = vol
    else:
        print("flower.off not found, skipping Task 1.4")
except Exception as e:
    print(f"Error in Task 1.4: {e}")

In [ ]:
# =============================================================================
# TASK 2: ICP Algorithm Setup
# =============================================================================

class ICP:
    def __init__(self, source_mesh, target_mesh, target_normals=None):
        self.source = self._to_numpy(source_mesh)
        self.target = self._to_numpy(target_mesh)
        self.target_normals = target_normals
        self.tree = KDTree(self.target)

    def _to_numpy(self, mesh):
        if hasattr(mesh, 'vertices'):
            return np.array([v.get_vertex() for v in mesh.vertices])
        return np.array(mesh)

    def find_nearest(self, points):
        dists, indices = self.tree.query(points, k=1)
        return dists.flatten(), indices.flatten()

# Load Data for Task 2
target_path = "bunnies/bun000_v2.ply"
source_path = "bunnies/bun045_v2.ply"
M1, M2 = None, None
M1_normals = None

if HAS_O3D and os.path.exists(target_path) and os.path.exists(source_path):
    print("Loading point clouds for ICP...")
    pcd_t = o3d.io.read_point_cloud(target_path)
    M1 = np.asarray(pcd_t.points)
    pcd_s = o3d.io.read_point_cloud(source_path)
    M2 = np.asarray(pcd_s.points)
    
    # Pre-calculate normals for Task 2.3
    pcd_t.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.01, max_nn=30))
    M1_normals = np.asarray(pcd_t.normals)
else:
    print("Warning: Data files for Task 2 not found or Open3D missing.")

In [ ]:
# =============================================================================
# =============================================================================


def run_point_to_point_icp(
    solver,
    max_iter=50,
    tol=1e-6,
    reject_percentile=90,
    verbose=False
):
    """
    Point-to-Point ICP using Kabsch + percentile rejection.
    NO centroid pre-alignment.
    """

    source = solver.source
    target = solver.target
    tree = solver.tree

    # Global transform (important!)
    R = np.eye(3)
    t = np.zeros(3)

    prev_err = None

    for it in range(max_iter):
        # Apply current transform
        src_tf = (source @ R.T) + t[None, :]

        # Nearest neighbors
        dists, indices = tree.query(src_tf, k=1)
        dists = dists.flatten()
        indices = indices.flatten()

        # Percentile rejection (KEY DIFFERENCE)
        thr = np.percentile(dists, reject_percentile)
        mask = dists <= thr

        if np.sum(mask) < 3:
            break

        P = src_tf[mask]
        Q = target[indices[mask]]

        # --- Kabsch ---
        Pc = P.mean(axis=0)
        Qc = Q.mean(axis=0)

        X = P - Pc
        Y = Q - Qc

        H = X.T @ Y
        U, _, Vt = np.linalg.svd(H)
        Ri = Vt.T @ U.T

        if np.linalg.det(Ri) < 0:
            Vt[-1, :] *= -1
            Ri = Vt.T @ U.T

        ti = Qc - Ri @ Pc

        # Compose transforms (GLOBAL accumulation)
        R = Ri @ R
        t = Ri @ t + ti

        # Evaluate RMSE on inliers
        src_tf2 = (source @ R.T) + t[None, :]
        d2, _ = tree.query(src_tf2, k=1)
        d2 = d2.flatten()
        thr2 = np.percentile(d2, reject_percentile)
        inlier2 = d2 <= thr2

        err = np.sqrt(np.mean(d2[inlier2] ** 2))

        if verbose:
            print(f"[ICP p2p] it={it:02d} rmse={err:.6f} inlier_ratio={inlier2.mean():.3f}")

        if prev_err is not None and abs(prev_err - err) < tol:
            break
        prev_err = err

    final_src = (source @ R.T) + t[None, :]
    return final_src, prev_err

In [ ]:
print("\n--- Task 2.1: Point-to-Point ICP ---")
solver = ICP(M2, M1)

visualize_superposition(M2, M1, "Task 2.1 - Start")

res_p2p, err = run_point_to_point_icp(
    solver,
    max_iter=50,
    reject_percentile=90,
    verbose=True
)

print(f"P2P Final RMSE: {err:.6f}")
visualize_superposition(res_p2p, M1, "Task 2.1 - End")

In [ ]:
# =============================================================================
# =============================================================================


def task_2_2a_rotation(M1, M2):
    print("\n--- Task 2.2a: Rotation Perturbation ---")
    angles = [0, 5, 10, 15, 20, 30]

    for angle in angles:
        theta = np.radians(angle)
        c, s = np.cos(theta), np.sin(theta)
        Rz = np.array([[c, -s, 0],
                       [s,  c, 0],
                       [0,  0, 1]])

        perturbed_M2 = (M2 @ Rz.T)

        visualize_superposition(
            perturbed_M2, M1,
            f"Task 2.2a (Rot={angle} deg) - Start"
        )

        solver = ICP(perturbed_M2, M1)
        aligned_pts, err = run_point_to_point_icp(
            solver,
            max_iter=60,
            reject_percentile=90,
            verbose=False
        )

        print(f"Rotation: {angle:3d} deg | Final RMSE: {err:.6f}")

        visualize_superposition(
            aligned_pts, M1,
            f"Task 2.2a (Rot={angle} deg) - End"
        )

# --- Execution for Task 2.2a ---
if M1 is not None and M2 is not None:
    task_2_2a_rotation(M1, M2)

In [ ]:
# =============================================================================
# =============================================================================



def task_2_2b_noise(M1, M2):
    print("\n--- Task 2.2b: Noise Perturbation ---")

    bbox_min = np.min(M2, axis=0)
    bbox_max = np.max(M2, axis=0)
    scale = np.linalg.norm(bbox_max - bbox_min)

    levels = [0.0, 0.01, 0.02, 0.05]

    for lvl in levels:
        noise = np.random.normal(0, scale * lvl, M2.shape)
        noisy_M2 = M2 + noise

        visualize_superposition(
            noisy_M2, M1,
            f"Task 2.2b (Noise={lvl*100:.0f}%) - Start"
        )

        solver = ICP(noisy_M2, M1)
        aligned_pts, err = run_point_to_point_icp(
            solver,
            max_iter=60,
            reject_percentile=90,
            verbose=False
        )

        print(f"Noise Level: {lvl*100:.0f}% | Final RMSE: {err:.6f}")

        visualize_superposition(
            aligned_pts, M1,
            f"Task 2.2b (Noise={lvl*100:.0f}%) - End"
        )

# --- Execution for Task 2.2b ---
if M1 is not None and M2 is not None:
    task_2_2b_noise(M1, M2)

In [ ]:
# =============================================================================
# TASK 2.3: Point-to-Plane ICP (Optimized Version)
# =============================================================================

def run_point_to_plane_icp(solver, max_iter=50, tol=1e-6, reject_percentile=90, verbose=False):
    if solver.target_normals is None: 
        print("Error: Target normals required for Point-to-Plane.")
        return solver.source
        
    source = solver.source
    target = solver.target
    normals = solver.target_normals
    tree = solver.tree

    # Global transform accumulation (Best Practice)
    R_global = np.eye(3)
    t_global = np.zeros(3)

    for i in range(max_iter):
        # Apply current global transform to ORIGINAL source points
        src_tf = (source @ R_global.T) + t_global[None, :]

        # Find nearest neighbors
        dists, indices = tree.query(src_tf, k=1)
        dists = dists.flatten()
        indices = indices.flatten()
        
        # Percentile rejection (Keep top X% to handle partial overlap)
        thr = np.percentile(dists, reject_percentile)
        mask = dists <= thr
        
        if np.sum(mask) < 3: break
        
        p = src_tf[mask]
        q = target[indices[mask]]
        n = normals[indices[mask]]
        
        # Construct Linear System for Point-to-Plane
        cross_pn = np.cross(p, n)
        A = np.hstack((cross_pn, n))
        b = np.sum((q - p) * n, axis=1)
        x, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        
        # Extract rotation and translation increments
        rx, ry, rz = x[:3]
        tx, ty, tz = x[3:]
        
        # Small angle approximation for rotation increment
        R_inc = np.array([
            [1, -rz, ry],
            [rz, 1, -rx],
            [-ry, rx, 1]
        ])
        
        # Orthogonalize to ensure valid rotation matrix
        U, _, Vt = np.linalg.svd(R_inc)
        R_inc = U @ Vt
        t_inc = np.array([tx, ty, tz])
        
        # Update Global Transform
        R_global = R_inc @ R_global
        t_global = R_inc @ t_global + t_inc
        
        # Check convergence
        update_norm = np.linalg.norm(x)
        if verbose:
            print(f"[ICP p2plane] it={i:02d} update_norm={update_norm:.6f}")
            
        if update_norm < tol: 
            break
            
    final_src = (source @ R_global.T) + t_global[None, :]
    return final_src



# --- Execution for Task 2.3 ---
if M1 is not None and M2 is not None and M1_normals is not None:
    print("\n--- Task 2.3: Point-to-Plane ICP ---")
    visualize_superposition(M2, M1, "Task 2.3 - Start")
    
    solver_plane = ICP(M2, M1, target_normals=M1_normals)
    res_p2pl = run_point_to_plane_icp(solver_plane, max_iter=50, reject_percentile=90, verbose=True)
    
    visualize_superposition(res_p2pl, M1, "Task 2.3 - End")